In [1]:
import numpy as np
import os
import sys
import astropy as ast
from astropy.io import ascii
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import scipy as sp
from astropy.timeseries import LombScargle
from astropy import units as u
import gc
# from view_and_clean import get_telescope, find_header_line, modified_zscore, df_extract, individual_plotter, grid_plotter, offset_corrector, offset_corrector_window
# from metrics import mean_med_flux, fit_plotter_flux, curve_fitter_flux,\
#     largest_amplitude, offset_warning, chi, percentile_amplitude, plot_percentile_amplitude, \
#         compute_lomb_scargle, plot_periodogram, plot_phase_fold, amplitude_per_period, amplitude_per_period_plot, sed_plotter

In [3]:
maria_spectroscopy = pd.read_csv('../ysg_candidates/original_files_from_anna/maria_ysgs_hms.csv')
original_stars = pd.read_csv('../merged_smc_lmc_coords.csv', comment='#', sep="\\s+", names=['ra', 'dec'])
prefinal_stars = pd.read_csv('../prefinal_merged_smc_lmc_coords.csv',comment='#', sep="\\s+", names=['ra', 'dec'])


In [3]:
def hms_to_deg(hms_str):
    """Convert RA from HH:MM:SS.sss to decimal degrees."""
    parts = hms_str.split(':')
    h = float(parts[0])
    m = float(parts[1])
    s = float(parts[2])
    return (h + m/60 + s/3600) * 15

def dms_to_deg(dms_str):
    """Convert Dec from DD:MM:SS.ss to decimal degrees."""
    parts = dms_str.split(':')
    d = float(parts[0])
    m = float(parts[1])
    s = float(parts[2])
    sign = -1 if d < 0 else 1
    return sign * (abs(d) + m/60 + s/3600)

In [4]:
maria_spectroscopy['ra_deg'] = maria_spectroscopy['ra'].apply(hms_to_deg)
maria_spectroscopy['dec_deg'] = maria_spectroscopy['dec'].apply(dms_to_deg)
print(f"Maria spectroscopy sample with converted coordinates:")
print(maria_spectroscopy.head())

Maria spectroscopy sample with converted coordinates:
             ra           dec           scope     ra_deg    dec_deg
0  04:49:14.025  -68:58:11.59  magellan_drout  72.308437 -68.969886
1  04:50:55.853  -69:25:52.50  magellan_drout  72.732721 -69.431250
2  04:51:45.106  -69:30:17.83  magellan_drout  72.937942 -69.504953
3  04:51:58.111  -69:25:33.84  magellan_drout  72.992129 -69.426067
4  04:52:11.299  -71:48:34.15  magellan_drout  73.047079 -71.809486


In [5]:
def angular_separation(ra1, dec1, ra2, dec2):
    """All inputs in decimal degrees."""
    coords1 = SkyCoord(ra=ra1*u.degree, dec=dec1*u.degree)
    coords2 = SkyCoord(ra=ra2*u.degree, dec=dec2*u.degree)
    sep = coords1.separation(coords2)
    return sep.arcsecond

def find_matches(maria_row, catalog_df, tolerance_arcsec=1.0):
    separations = angular_separation(maria_row['ra_deg'], maria_row['dec_deg'], catalog_df['ra'].values, catalog_df['dec'].values)
    matches = separations < tolerance_arcsec
    match_indices = np.where(matches)[0]
    match_seps = separations[matches]
    return match_indices, match_seps

In [6]:
# Load the summary file with star_idx and coordinates
# summary_file = pd.read_csv('synth_phot_temp_estimation/ysg_temp_fitting_summary_v10_prefinal.csv')
summary_file = pd.read_csv('summary_results03092026.csv')
print(f"Loaded summary file with {len(summary_file)} stars")
print(f"  Original stars (star_idx < 848): {(summary_file['star_idx'] < 848).sum()}")
print(f"  Prefinal stars (star_idx >= 848): {(summary_file['star_idx'] >= 848).sum()}")

Loaded summary file with 1270 stars
  Original stars (star_idx < 848): 848
  Prefinal stars (star_idx >= 848): 422


In [7]:
tolerance_arcsec = 1.0 

results = []
for idx, row in maria_spectroscopy.iterrows():
    # Find matches
    orig_matches, orig_seps = find_matches(row, original_stars, tolerance_arcsec)
    prefinal_matches, prefinal_seps = find_matches(row, prefinal_stars, tolerance_arcsec)
    
    result = {
        'scope': row['scope'],
        'ra_hms': row['ra'],
        'dec_dms': row['dec'],
        'ra_deg': row['ra_deg'],
        'dec_deg': row['dec_deg'],
        'in_original': len(orig_matches) > 0,
        'in_prefinal': len(prefinal_matches) > 0,
    }
    
    if len(orig_matches) > 0:
        best_idx = orig_matches[np.argmin(orig_seps)]
        match_star = summary_file.iloc[best_idx]
        result['original_star_idx'] = int(match_star['star_idx'])
        result['original_ra'] = match_star['RA']
        result['original_dec'] = match_star['DEC']
        result['original_sep_arcsec'] = orig_seps[np.argmin(orig_seps)]
    else:
        result['original_star_idx'] = np.nan
        result['original_ra'] = np.nan
        result['original_dec'] = np.nan
        result['original_sep_arcsec'] = np.nan
    
    if len(prefinal_matches) > 0:
        best_idx = prefinal_matches[np.argmin(prefinal_seps)]
        match_star = summary_file.iloc[848 + best_idx]  # Offset by original stars
        result['prefinal_star_idx'] = int(match_star['star_idx'])
        result['prefinal_ra'] = match_star['RA']
        result['prefinal_dec'] = match_star['DEC']
        result['prefinal_sep_arcsec'] = prefinal_seps[np.argmin(prefinal_seps)]
    else:
        result['prefinal_star_idx'] = np.nan
        result['prefinal_ra'] = np.nan
        result['prefinal_dec'] = np.nan
        result['prefinal_sep_arcsec'] = np.nan
    
    results.append(result)

match_results = pd.DataFrame(results)
print(f"Total stars: {len(match_results)}")
print(f"Matched to original: {match_results['in_original'].sum()}")
print(f"Matched to prefinal: {match_results['in_prefinal'].sum()}")
print(f"Matched to both: {(match_results['in_original'] & match_results['in_prefinal']).sum()}")
print(f"Not matched: {(~match_results['in_original'] & ~match_results['in_prefinal']).sum()}")

Total stars: 125
Matched to original: 37
Matched to prefinal: 74
Matched to both: 0
Not matched: 14


In [8]:
print(match_results[['scope', 'ra_hms', 'dec_dms', 'in_original', 'original_star_idx', 'in_prefinal', 'prefinal_star_idx']].head(10))

            scope        ra_hms       dec_dms  in_original  original_star_idx  \
0  magellan_drout  04:49:14.025  -68:58:11.59        False                NaN   
1  magellan_drout  04:50:55.853  -69:25:52.50         True              403.0   
2  magellan_drout  04:51:45.106  -69:30:17.83         True              407.0   
3  magellan_drout  04:51:58.111  -69:25:33.84        False                NaN   
4  magellan_drout  04:52:11.299  -71:48:34.15        False                NaN   
5  magellan_drout  04:52:53.379  -67:05:42.58         True              415.0   
6  magellan_drout  04:54:14.257  -69:12:36.44        False                NaN   
7  magellan_drout  04:54:36.840  -69:20:22.10        False                NaN   
8  magellan_drout  04:54:57.350  -66:45:08.83         True              433.0   
9  magellan_drout  04:55:10.089  -66:50:42.55        False                NaN   

   in_prefinal  prefinal_star_idx  
0         True             1014.0  
1        False                NaN  


# write to files here:

In [9]:
# # Stars matched to original
# original_mask = match_results['in_original']
# if original_mask.sum() > 0:
#     match_results[original_mask].to_csv('magellan_spectroscopy_to_original_matches.csv', index=False)

# # Stars matched to prefinal
# prefinal_mask = match_results['in_prefinal']
# if prefinal_mask.sum() > 0:
#     match_results[prefinal_mask].to_csv('magellan_spectroscopy_to_prefinal_matches.csv', index=False)

# # Stars not matched to either
# no_match_mask = ~match_results['in_original'] & ~match_results['in_prefinal']
# if no_match_mask.sum() > 0:
#     match_results[no_match_mask].to_csv('magellan_spectroscopy_no_matches.csv', index=False)

# double check using same method but against MIKE LMC

In [10]:
mike_spectroscopy = pd.read_csv('ysg_candidates/original_files_from_anna/MIKE_LMC.csv')
mike_spectroscopy['ra_deg'] = mike_spectroscopy['RA'].apply(hms_to_deg)
mike_spectroscopy['dec_deg'] = mike_spectroscopy['DEC'].apply(dms_to_deg)
print(len(mike_spectroscopy))

108


In [11]:
def angular_separation(ra1, dec1, ra2, dec2):
    """All inputs in decimal degrees."""
    coords1 = SkyCoord(ra=ra1*u.degree, dec=dec1*u.degree)
    coords2 = SkyCoord(ra=ra2*u.degree, dec=dec2*u.degree)
    sep = coords1.separation(coords2)
    return sep.arcsecond

def find_matches(mike_row, catalog_df, tolerance_arcsec=1.0):
    separations = angular_separation(mike_row['ra_deg'], mike_row['dec_deg'], catalog_df['ra'].values, catalog_df['dec'].values)
    matches = separations < tolerance_arcsec
    match_indices = np.where(matches)[0]
    match_seps = separations[matches]
    
    return match_indices, match_seps

In [12]:
tolerance_arcsec = 1.0

results = []
for idx, row in mike_spectroscopy.iterrows():
    # Find matches
    orig_matches, orig_seps = find_matches(row, original_stars, tolerance_arcsec)
    prefinal_matches, prefinal_seps = find_matches(row, prefinal_stars, tolerance_arcsec)
    
    result = {
        'ra_hms': row['RA'],
        'dec_dms': row['DEC'],
        'in_original': len(orig_matches) > 0,
        'in_prefinal': len(prefinal_matches) > 0,
    }
    
    if len(orig_matches) > 0:
        best_idx = orig_matches[np.argmin(orig_seps)]
        match_star = summary_file.iloc[best_idx]
        result['original_star_idx'] = int(match_star['star_idx'])
        result['original_ra'] = match_star['RA']
        result['original_dec'] = match_star['DEC']
        result['original_sep_arcsec'] = orig_seps[np.argmin(orig_seps)]
    else:
        result['original_star_idx'] = np.nan
        result['original_ra'] = np.nan
        result['original_dec'] = np.nan
        result['original_sep_arcsec'] = np.nan
    
    # Add prefinal catalog match info
    if len(prefinal_matches) > 0:
        best_idx = prefinal_matches[np.argmin(prefinal_seps)]
        match_star = summary_file.iloc[848 + best_idx]  # Offset by original stars
        result['prefinal_star_idx'] = int(match_star['star_idx'])
        result['prefinal_ra'] = match_star['RA']
        result['prefinal_dec'] = match_star['DEC']
        result['prefinal_sep_arcsec'] = prefinal_seps[np.argmin(prefinal_seps)]
    else:
        result['prefinal_star_idx'] = np.nan
        result['prefinal_ra'] = np.nan
        result['prefinal_dec'] = np.nan
        result['prefinal_sep_arcsec'] = np.nan
    
    results.append(result)

match_results = pd.DataFrame(results)
print(f"Total stars: {len(match_results)}")
print(f"Matched to original: {match_results['in_original'].sum()}")
print(f"Matched to prefinal: {match_results['in_prefinal'].sum()}")
print(f"Not matched: {(~match_results['in_original'] & ~match_results['in_prefinal']).sum()}")

Total stars: 108
Matched to original: 37
Matched to prefinal: 62
Not matched: 9


# sorting out downloaded gaia bprp synthetic photometry file:

In [13]:
gaia_bprp = pd.read_csv('gaia_bprp_synthphot.csv', sep=';', comment='#') #, skiprows=[1, 2])
# Drop the first 2 rows which contain units and separator lines
gaia_bprp = gaia_bprp.iloc[2:].reset_index(drop=True)
# Convert ra and dec to numeric
gaia_bprp['ra'] = pd.to_numeric(gaia_bprp['ra'], errors='coerce')
gaia_bprp['dec'] = pd.to_numeric(gaia_bprp['dec'], errors='coerce')
print(f"Loaded {len(gaia_bprp)} stars from Gaia")

Loaded 1270 stars from Gaia


In [14]:
print(gaia_bprp.head())
print(len(gaia_bprp))

         ra        dec     _r               Source          RA_ICRS  \
0  7.521702 -73.913918  0.161  4688433892854787328    7.52180281079   
1  7.746443 -73.741425  0.052  4688451862997785728    7.74648572404   
2  7.925977 -73.531670  0.031  4688458185189527040    7.92597456883   
3  7.948884 -73.599243  0.026  4688456810800008832    7.94888660474   
4  7.980047 -73.578545  0.041  4688456913879215488    7.98003768557   

           DE_ICRS E(BP/RP)corr            FU          e_FU       Umag  ...  \
0  -73.91388314441      -0.0021   8.04136e-17   8.53155e-19  14.387775  ...   
1  -73.74141704632      -0.0010   6.39884e-17   8.11467e-19  14.635846  ...   
2  -73.53167859623      -0.0032                                         ...   
3  -73.59925025421      -0.0089                                         ...   
4  -73.57855596015      -0.0116                                         ...   

        gmag gFlag            Fr          e_Fr       rmag rFlag            Fi  \
0  13.854267     

In [49]:
print("Columns in gaia_bprp:")
print(gaia_bprp.columns.tolist())
print("\nFirst few rows of first 5 columns:")
print(gaia_bprp.iloc[:, :5].head())

Columns in gaia_bprp:
['ra', 'dec', '_r', 'Source', 'RA_ICRS', 'DE_ICRS', 'E(BP/RP)corr', 'FU', 'e_FU', 'Umag', 'UFlag', 'FB', 'e_FB', 'Bmag', 'BFlag', 'FV', 'e_FV', 'Vmag', 'VFlag', 'FR', 'e_FR', 'Rmag', 'RFlag', 'FI', 'e_FI', 'Imag', 'IFlag', 'Fu', 'e_Fu', 'umag', 'uFlag', 'Fg', 'e_Fg', 'gmag', 'gFlag', 'Fr', 'e_Fr', 'rmag', 'rFlag', 'Fi', 'e_Fi', 'imag', 'iFlag']

First few rows of first 5 columns:
         ra        dec     _r               Source          RA_ICRS
0  7.521702 -73.913918  0.161  4688433892854787328    7.52180281079
1  7.746443 -73.741425  0.052  4688451862997785728    7.74648572404
2  7.925977 -73.531670  0.031  4688458185189527040    7.92597456883
3  7.948884 -73.599243  0.026  4688456810800008832    7.94888660474
4  7.980047 -73.578545  0.041  4688456913879215488    7.98003768557


In [16]:
########## TO SEPARATE INPUT RA AND DEC FROM SEMICOLON-SEPARATED FILES, NO LONGER NEEDED ############

# with open('gaia_bprp_synth_phot.csv', 'r') as f:
#     lines = f.readlines()

# # Process each data line (after line 47)
# for i in range(47, len(lines)):
#     # Find the first semicolon
#     semicolon_pos = lines[i].find(';')
#     if semicolon_pos > 0:
#         # Get the coordinate part
#         coord_part = lines[i][:semicolon_pos].strip()
#         # Split on whitespace into exactly 2 parts
#         parts = coord_part.split(None, 1)
#         if len(parts) == 2:
#             # Reconstruct with semicolon separator
#             lines[i] = f"{parts[0]};{parts[1]};{lines[i][semicolon_pos+1:]}"

# with open('gaia_bprp_synthphot.csv', 'w') as f:
#     f.writelines(lines)

In [17]:
# checking how many have B and V photometry:
gaia_bprp_B = gaia_bprp[gaia_bprp['Bmag'].notna()]
print(f"Number of stars with B photometry: {len(gaia_bprp_B)}")
gaia_bprp_V = gaia_bprp[gaia_bprp['Vmag'].notna()]
print(f"Number of stars with V photometry: {len(gaia_bprp_V)}")
gaia_bprp_BV = gaia_bprp[gaia_bprp['Bmag'].notna() & gaia_bprp['Vmag'].notna()]
print(f"Number of stars with both B and V photometry: {len(gaia_bprp_BV)}")
gaia_bprp_g = gaia_bprp[gaia_bprp['gmag'].notna()]
print(f"Number of stars with g photometry: {len(gaia_bprp_g)}")

print(gaia_bprp['Bmag'].isna().sum())
for star in gaia_bprp[gaia_bprp['Bmag'].isna()]:
    print(gaia_bprp[gaia_bprp['Bmag'].isna()])
# print(gaia_bprp['Bmag'].isna())

Number of stars with B photometry: 1249
Number of stars with V photometry: 1249
Number of stars with both B and V photometry: 1249
Number of stars with g photometry: 1249
21
             ra        dec   _r Source RA_ICRS DE_ICRS E(BP/RP)corr   FU e_FU  \
143   13.350006 -73.110420  NaN    NaN     NaN     NaN          NaN  NaN  NaN   
477   74.527298 -67.241653  NaN    NaN     NaN     NaN          NaN  NaN  NaN   
520   76.059927 -66.431381  NaN    NaN     NaN     NaN          NaN  NaN  NaN   
792   85.507777 -68.714325  NaN    NaN     NaN     NaN          NaN  NaN  NaN   
852    8.648353 -72.544128  NaN    NaN     NaN     NaN          NaN  NaN  NaN   
862   11.325652 -73.256996  NaN    NaN     NaN     NaN          NaN  NaN  NaN   
1027  74.464499 -69.394508  NaN    NaN     NaN     NaN          NaN  NaN  NaN   
1030  72.288057 -68.959038  NaN    NaN     NaN     NaN          NaN  NaN  NaN   
1046  76.037142 -70.262787  NaN    NaN     NaN     NaN          NaN  NaN  NaN   
1070  77.189406 

In [18]:
# crossmatch to maria_spectroscopy
def angular_separation(ra1, dec1, ra2, dec2):
    """All inputs in decimal degrees."""
    coords1 = SkyCoord(ra=ra1*u.degree, dec=dec1*u.degree)
    coords2 = SkyCoord(ra=ra2*u.degree, dec=dec2*u.degree)
    sep = coords1.separation(coords2)
    return sep.arcsecond

def find_matches(maria_row, catalog_df, tolerance_arcsec=1.0):
    separations = angular_separation(maria_row['ra_deg'], maria_row['dec_deg'], catalog_df['ra'].values, catalog_df['dec'].values)
    matches = separations < tolerance_arcsec
    match_indices = np.where(matches)[0]
    match_seps = separations[matches]
    return match_indices, match_seps

tolerance_arcsec = 1.0
results = []
for idx, row in maria_spectroscopy.iterrows():
    # Find matches
    gaia_matches, gaia_seps = find_matches(row, gaia_bprp, tolerance_arcsec)
    
    result = {
        'ra_hms': row['ra'],
        'dec_dms': row['dec'],
        'in_gaia': len(gaia_matches) > 0,
    }
    
    if len(gaia_matches) > 0:
        best_idx = gaia_matches[np.argmin(gaia_seps)]
        match_star = gaia_bprp.iloc[best_idx]
        # result['gaia_star_idx'] = int(match_star['star_idx'])
        result['gaia_ra'] = match_star['ra']
        result['gaia_dec'] = match_star['dec']
        result['gaia_sep_arcsec'] = gaia_seps[np.argmin(gaia_seps)]
    else:
        # result['gaia_star_idx'] = np.nan
        result['gaia_ra'] = np.nan
        result['gaia_dec'] = np.nan
        result['gaia_sep_arcsec'] = np.nan

    
    results.append(result)

match_results_spectra_to_gaia = pd.DataFrame(results)
print(match_results_spectra_to_gaia.head())

print(f"Total stars in spectroscopy file: {len(match_results_spectra_to_gaia)}")
print(f"Matched to gaia: {match_results_spectra_to_gaia['in_gaia'].sum()}")
print(f"Not matched: {(~match_results_spectra_to_gaia['in_gaia']).sum()}")
print(match_results_spectra_to_gaia['in_gaia'][match_results_spectra_to_gaia['in_gaia'] == False])

         ra_hms       dec_dms  in_gaia    gaia_ra   gaia_dec  gaia_sep_arcsec
0  04:49:14.025  -68:58:11.59     True  72.308488 -68.969910         0.107946
1  04:50:55.853  -69:25:52.50     True  72.732719 -69.431274         0.086431
2  04:51:45.106  -69:30:17.83     True  72.937831 -69.504936         0.152006
3  04:51:58.111  -69:25:33.84     True  72.992118 -69.426094         0.099409
4  04:52:11.299  -71:48:34.15    False        NaN        NaN              NaN
Total stars in spectroscopy file: 125
Matched to gaia: 111
Not matched: 14
4      False
12     False
18     False
56     False
62     False
63     False
70     False
74     False
77     False
112    False
120    False
122    False
123    False
124    False
Name: in_gaia, dtype: bool


# crossmatch the missing stars in gaia spectra to temp estimations to see which stars they are and if we have temperatures already for them.

In [47]:
# crossmatch the missing stars in gaia spectra to summary file to see which stars they are and if we have temperatures already for them.
temps = pd.read_csv('synth_phot_temp_estimation/ysg_temp_fitting_summary_03092026.csv')
no_temps = temps[temps['final_teff_mean'].isna()]
print(f"Number of stars with missing temperatures due to a lack of optical data (prior to checking gaia): {len(no_temps)}")

missing_from_gaia = gaia_bprp[gaia_bprp['Bmag'].isna()]
print(f"number of stars from total list (1270) that did not match to gaia: {len(missing_from_gaia)}")

def angular_separation(ra1, dec1, ra2, dec2):
    """All inputs in decimal degrees."""
    coords1 = SkyCoord(ra=ra1*u.degree, dec=dec1*u.degree)
    coords2 = SkyCoord(ra=ra2*u.degree, dec=dec2*u.degree)
    sep = coords1.separation(coords2)
    return sep.arcsecond

def find_matches(spec_row, catalog_df, tolerance_arcsec=1.0):
    separations = angular_separation(spec_row['ra'], spec_row['dec'], catalog_df['RA'].values, catalog_df['DEC'].values)
    matches = separations < tolerance_arcsec
    match_indices = np.where(matches)[0]
    match_seps = separations[matches]
    return match_indices, match_seps

tolerance_arcsec = 1.0
results = []

for idx, row in missing_from_gaia.iterrows():
    # Find matches to stars with no temps
    no_temp_matches, no_temp_seps = find_matches(row, no_temps, tolerance_arcsec)
    
    result = {
        'ra': row['ra'],                              # Keep original coordinates
        'dec': row['dec'],                            # Keep original coordinates
        'matched_to_no_temps': len(no_temp_matches) > 0,
    }
    
    if len(no_temp_matches) > 0:
        best_idx = no_temp_matches[np.argmin(no_temp_seps)]
        match_star = no_temps.iloc[best_idx]         # Get the matched star from no_temps
        result['nooptical_star_idx'] = int(match_star['star_idx'])
        # result['matched_star_ra'] = match_star['RA']
        # result['matched_star_dec'] = match_star['DEC']
        # result['separation_arcsec'] = no_temp_seps[np.argmin(no_temp_seps)]
    else:
        # result['nooptical_star_idx'] = np.nan
        # result['matched_star_ra'] = np.nan
        # result['matched_star_dec'] = np.nan
        # result['separation_arcsec'] = np.nan
        result['nooptical_star_idx'] = np.nan
    
    results.append(result)

match_results_nogaia_nooptical = pd.DataFrame(results)
print(f"Of those, matched to stars with no temps: {match_results_nogaia_nooptical['matched_to_no_temps'].sum()}")


Number of stars with missing temperatures due to a lack of optical data (prior to checking gaia): 153
number of stars from total list (1270) that did not match to gaia: 21
Of those, matched to stars with no temps: 11


In [50]:
# now let's check the stars that are missing from gaia AND no temps to see if they are in maria spectroscopy
# Filter to only stars that matched to no_temps (these are the ones lacking both Gaia AND temperatures)
stars_nogaia_and_notemps = match_results_nogaia_nooptical[match_results_nogaia_nooptical['matched_to_no_temps'] == True]
print(f"Stars missing BOTH Gaia and temperatures: {len(stars_nogaia_and_notemps)}")
#write to file
# stars_nogaia_and_notemps.to_csv('stars_missing_gaia_and_temps.csv', index=False)

# def angular_separation(ra1, dec1, ra2, dec2):
#     """All inputs in decimal degrees."""
#     coords1 = SkyCoord(ra=ra1*u.degree, dec=dec1*u.degree)
#     coords2 = SkyCoord(ra=ra2*u.degree, dec=dec2*u.degree)
#     sep = coords1.separation(coords2)
#     return sep.arcsecond

# def find_matches(row, catalog_df, tolerance_arcsec=1.0):
#     # maria_spectroscopy has ra_deg and dec_deg, not ra and dec
#     separations = angular_separation(row['ra'], row['dec'], catalog_df['ra_deg'].values, catalog_df['dec_deg'].values)
#     matches = separations < tolerance_arcsec
#     match_indices = np.where(matches)[0]
#     match_seps = separations[matches]
#     return match_indices, match_seps

# tolerance_arcsec = 1.0
# results = []

# # Loop through only stars that are missing BOTH Gaia and temperatures
# for idx, row in stars_nogaia_and_notemps.iterrows():
#     # Find matches to maria_spectroscopy
#     spec_matches, spec_seps = find_matches(row, maria_spectroscopy, tolerance_arcsec)
    
#     result = {
#         'ra': row['ra'],                              # Keep original coordinates
#         'dec': row['dec'],                            # Keep original coordinates
#         'star_idx': row['matched_star_idx'],          # From the no_temps match
#         'matched_to_spectroscopy': len(spec_matches) > 0,
#     }
    
#     if len(spec_matches) > 0:
#         best_idx = spec_matches[np.argmin(spec_seps)]
#         match_star = maria_spectroscopy.iloc[best_idx]         # Get the matched star from maria_spectroscopy
#         result['matched_scope'] = match_star['scope']
#         result['matched_ra_hms'] = match_star['ra']
#         result['matched_dec_dms'] = match_star['dec']
#         result['matched_ra_deg'] = match_star['ra_deg']
#         result['matched_dec_deg'] = match_star['dec_deg']
#         result['separation_arcsec'] = spec_seps[np.argmin(spec_seps)]
#     else:
#         result['matched_scope'] = np.nan
#         result['matched_ra_hms'] = np.nan
#         result['matched_dec_dms'] = np.nan
#         result['matched_ra_deg'] = np.nan
#         result['matched_dec_deg'] = np.nan
#         result['separation_arcsec'] = np.nan
    
#     results.append(result)

# match_results_spectroscopy = pd.DataFrame(results)
# print(f"\nOf stars missing both Gaia AND temperatures:")
# print(f"  Total checked: {len(match_results_spectroscopy)}")
# print(f"  Matched to maria_spectroscopy: {match_results_spectroscopy['matched_to_spectroscopy'].sum()}")
# print(f"  Not matched to maria_spectroscopy: {(~match_results_spectroscopy['matched_to_spectroscopy']).sum()}")

Stars missing BOTH Gaia and temperatures: 11


## Working with UBVI gaia mags

In [62]:
gaia_bprp = pd.read_csv('./synth_phot_temp_estimation/gaia_bprp_synthphot.csv', sep=';', comment='#')
gaia_bprp = gaia_bprp.iloc[2:].reset_index(drop=True)

band_zeropoints_gaia = {
    'FU': 3.49719e-9,  # U-band
    'FB': 6.72553e-9,  # B-band
    'FV': 3.5833e-9,   # V-band
    'FI': 9.23651e-10, # I-band
    # 'Fg': 5.45476e-9,  # g-band
    # 'Fr': 2.49767e-9,  # r-band
}

output_col_map = {
    'FU': 'Umag_GAIA',
    'FB': 'Bmag_GAIA',
    'FV': 'Vmag_GAIA',
    'FI': 'Imag_GAIA',
    # 'Fg': 'gmag_GAIA',
    # 'Fr': 'rmag_GAIA',
}

for band, zeropoint in band_zeropoints_gaia.items():
    flux_col = band
    err_col = f'e_{band}'

    if flux_col not in gaia_bprp.columns:
        print(f"Skipping {band}: missing column '{flux_col}'")
        gaia_bprp[output_col_map[band]] = np.nan
        gaia_bprp[f"e_{output_col_map[band]}"] = np.nan
        continue

    flux_w_m2_nm = pd.to_numeric(gaia_bprp[flux_col], errors='coerce')
    flux_err_w_m2_nm = pd.to_numeric(gaia_bprp[err_col], errors='coerce') if err_col in gaia_bprp.columns else pd.Series(np.nan, index=gaia_bprp.index)

    # Convert W/m^2/nm -> erg/s/cm^2/A (factor 100)
    if band in ['FU', 'FB', 'FV', 'FI']:
        flux_erg = flux_w_m2_nm * 100.0
        flux_err_erg = flux_err_w_m2_nm * 100.0
    else:
        flux_erg = flux_w_m2_nm 
        flux_err_erg = flux_err_w_m2_nm 

    mag = pd.Series(np.nan, index=gaia_bprp.index, dtype=float)
    mag_err = pd.Series(np.nan, index=gaia_bprp.index, dtype=float)

    valid_flux = flux_erg > 0
    mag.loc[valid_flux] = -2.5 * np.log10(flux_erg.loc[valid_flux] / zeropoint)

    valid_err = valid_flux & flux_err_erg.notna()
    mag_err.loc[valid_err] = (2.5 / np.log(10)) * (flux_err_erg.loc[valid_err] / flux_erg.loc[valid_err])

    out_mag_col = output_col_map[band]
    out_err_col = f'e_{out_mag_col}'

    gaia_bprp[out_mag_col] = mag
    gaia_bprp[out_err_col] = mag_err

gaia_bprp.to_csv('gaia_bprp_synthphot_allbands.csv', sep=';', index=False)

new_cols = [
    'Umag_GAIA', 'e_Umag_GAIA',
    'Bmag_GAIA', 'e_Bmag_GAIA',
    'Vmag_GAIA', 'e_Vmag_GAIA',
    'Imag_GAIA', 'e_Imag_GAIA',
    # 'gmag_GAIA', 'e_gmag_GAIA',
    # 'rmag_GAIA', 'e_rmag_GAIA',
]

print('Wrote Gaia magnitude columns to gaia_bprp_synthphot.csv')
print(gaia_bprp[new_cols].head())

Wrote Gaia magnitude columns to gaia_bprp_synthphot.csv
   Umag_GAIA  e_Umag_GAIA  Bmag_GAIA  e_Bmag_GAIA  Vmag_GAIA  e_Vmag_GAIA  \
0  14.095974     0.011519  14.210405     0.002504  13.679211     0.002030   
1  14.344045     0.013769  14.469780     0.002637  13.881211     0.002074   
2        NaN          NaN  15.809591     0.005391  14.403441     0.002243   
3        NaN          NaN  15.948498     0.005433  14.474386     0.002548   
4        NaN          NaN  16.156027     0.006423  14.548944     0.002614   

   Imag_GAIA  e_Imag_GAIA  
0  12.842918     0.001430  
1  12.974329     0.001486  
2  12.824772     0.001459  
3  12.891754     0.001311  
4  12.874905     0.001272  


## Working with gri mags

In [70]:
gaia_bprp = pd.read_csv('./synth_phot_temp_estimation/gaia_bprp_synthphot.csv', sep=';', comment='#')
gaia_bprp = gaia_bprp.iloc[2:].reset_index(drop=True)

band_zeropoints_gaia = {
    # 'FU': 3.49719e-9,  # U-band
    # 'FB': 6.72553e-9,  # B-band
    # 'FV': 3.5833e-9,   # V-band
    # 'FI': 9.23651e-10, # I-band
    'Fg': 5.45476e-9,  # g-band
    'Fr': 2.49767e-9,  # r-band
    'Fi':1.38589e-9,  # i-band

}

band_wavelengths_gaia = {
    'Fg': 4671.78,
    'Fr': 6141.12,
    'Fi': 7457.89
}

output_col_map = {
    # 'FU': 'Umag_GAIA',
    # 'FB': 'Bmag_GAIA',
    # 'FV': 'Vmag_GAIA',
    # 'FI': 'Imag_GAIA',
    'Fg': 'gmag_GAIA',
    'Fr': 'rmag_GAIA',
    'Fi': 'imag_GAIA'
}

for band, zeropoint in band_zeropoints_gaia.items():
    flux_col = band
    err_col = f'e_{band}'

    if flux_col not in gaia_bprp.columns:
        print(f"Skipping {band}: missing column '{flux_col}'")
        gaia_bprp[output_col_map[band]] = np.nan
        gaia_bprp[f"e_{output_col_map[band]}"] = np.nan
        continue

    # this is W/m^2/Hz (F_nu), need to convert to W/m^2/nm (F_lambda)
    flux_w_m2_Hz = pd.to_numeric(gaia_bprp[flux_col], errors='coerce')
    flux_err_w_m2_Hz = pd.to_numeric(gaia_bprp[err_col], errors='coerce') if err_col in gaia_bprp.columns else pd.Series(np.nan, index=gaia_bprp.index)

    flux_w_m2_m = flux_w_m2_Hz * (3e8 / ((band_wavelengths_gaia[band]*1E-10)**2))  # F_lambda = F_nu * c / lambda^2
    flux_err_w_m2_m = flux_err_w_m2_Hz * (3e8 / ((band_wavelengths_gaia[band]*1E-10)**2))  # Propagate error with same conversion

    # Convert W/m^2/m -> erg/s/cm^2/A (factor 100)

    flux_erg = flux_w_m2_m * 1e-7  # W/m^2/m to erg/s/cm^2/A
    flux_err_erg = flux_err_w_m2_m * 1e-7

    mag = pd.Series(np.nan, index=gaia_bprp.index, dtype=float)
    mag_err = pd.Series(np.nan, index=gaia_bprp.index, dtype=float)

    valid_flux = flux_erg > 0
    mag.loc[valid_flux] = -2.5 * np.log10(flux_erg.loc[valid_flux] / zeropoint)

    valid_err = valid_flux & flux_err_erg.notna()
    mag_err.loc[valid_err] = (2.5 / np.log(10)) * (flux_err_erg.loc[valid_err] / flux_erg.loc[valid_err])

    out_mag_col = output_col_map[band]
    out_err_col = f'e_{out_mag_col}'
    out_flux_col = f'{out_mag_col}_fluxergs'
    out_flux_err_col = f'e_{out_flux_col}'


    gaia_bprp[out_mag_col] = mag
    gaia_bprp[out_err_col] = mag_err
    gaia_bprp[out_flux_col] = flux_erg
    gaia_bprp[out_flux_err_col] = flux_err_erg

gaia_bprp.to_csv('gaia_bprp_synthphot_allbands.csv', sep=';', index=False)

new_cols = [
    # 'Umag_GAIA', 'e_Umag_GAIA',
    # 'Bmag_GAIA', 'e_Bmag_GAIA',
    # 'Vmag_GAIA', 'e_Vmag_GAIA',
    # 'Imag_GAIA', 'e_Imag_GAIA',
    'gmag_GAIA', 'e_gmag_GAIA',
    'gmag_GAIA_fluxergs', 'e_gmag_GAIA_fluxergs',
    'rmag_GAIA', 'e_rmag_GAIA',
    'rmag_GAIA_fluxergs', 'e_rmag_GAIA_fluxergs',
    'imag_GAIA', 'e_imag_GAIA',
    'imag_GAIA_fluxergs', 'e_imag_GAIA_fluxergs'

]

print('Wrote Gaia magnitude columns to gaia_bprp_synthphot_allbands.csv')
print(gaia_bprp[new_cols].head())

Wrote Gaia magnitude columns to gaia_bprp_synthphot_allbands.csv
   gmag_GAIA  e_gmag_GAIA  gmag_GAIA_fluxergs  e_gmag_GAIA_fluxergs  \
0  15.461133     0.004232        3.567158e-15          1.390331e-17   
1  14.799358     0.005067        6.561944e-15          3.062139e-17   
2  14.626221     0.018034        7.696391e-15          1.278390e-16   
3  15.142085     0.003838        4.785657e-15          1.691492e-17   
4  13.268392     0.001952        2.687935e-14          4.833161e-17   

   rmag_GAIA  e_rmag_GAIA  rmag_GAIA_fluxergs  e_rmag_GAIA_fluxergs  \
0  13.943734     0.001895        6.607566e-15          1.153302e-17   
1  13.271611     0.002033        1.227130e-14          2.297703e-17   
2  13.919534     0.012390        6.756495e-15          7.710102e-17   
3  13.734979     0.001792        8.008357e-15          1.322118e-17   
4  13.128158     0.001851        1.400464e-14          2.387138e-17   

   imag_GAIA  e_imag_GAIA  imag_GAIA_fluxergs  e_imag_GAIA_fluxergs  
0  13.26574

In [61]:
# print difference between original B and V mags and the new ones we calculated from fluxes for the stars that have both
gaia_bprp_BV = gaia_bprp[gaia_bprp['Bmag_GAIA'].notna() & gaia_bprp['Vmag_GAIA'].notna()]
print(f"Number of stars with both B and V photometry (newly calculated): {len(gaia_bprp_BV)}")
if len(gaia_bprp_BV) > 0:
    gaia_bprp_BV['Bmag_diff'] = gaia_bprp_BV['Bmag_GAIA'] - pd.to_numeric(gaia_bprp_BV['Bmag'], errors='coerce')
    gaia_bprp_BV['Vmag_diff'] = gaia_bprp_BV['Vmag_GAIA'] - pd.to_numeric(gaia_bprp_BV['Vmag'], errors='coerce')
    print(gaia_bprp_BV[['Bmag', 'Bmag_GAIA', 'Bmag_diff', 'Vmag', 'Vmag_GAIA', 'Vmag_diff']].head())

# gaia_bprp_gr = gaia_bprp[gaia_bprp['gmag_GAIA'].notna() & gaia_bprp['rmag_GAIA'].notna()]
# print(f"Number of stars with both g and r photometry (newly calculated): {len(gaia_bprp_gr)}")
# if len(gaia_bprp_gr) > 0:
#     gaia_bprp_gr['gmag_diff'] = gaia_bprp_gr['gmag_GAIA'] - pd.to_numeric(gaia_bprp_gr['gmag'], errors='coerce')
#     gaia_bprp_gr['rmag_diff'] = gaia_bprp_gr['rmag_GAIA'] - pd.to_numeric(gaia_bprp_gr['rmag'], errors='coerce')
#     print(gaia_bprp_gr[['gmag', 'gmag_GAIA', 'gmag_diff', 'rmag', 'rmag_GAIA', 'rmag_diff']].head())

for row in gaia_bprp.itertuples():
    g_diff = row.gmag_GAIA - pd.to_numeric(row.gmag, errors='coerce') if pd.notna(row.gmag_GAIA) and pd.notna(row.gmag) else np.nan
    B_diff = row.Bmag_GAIA - pd.to_numeric(row.Bmag, errors='coerce') if pd.notna(row.Bmag_GAIA) and pd.notna(row.Bmag) else np.nan
    print(f"Original B: {row.Bmag}, Calculated B: {row.Bmag_GAIA}, Diff: {B_diff}")
    print(f"Original g: {row.gmag}, Calculated g: {row.gmag_GAIA}, Diff: {g_diff}")
    print("-----")

Number of stars with both B and V photometry (newly calculated): 1177
        Bmag  Bmag_GAIA  Bmag_diff       Vmag  Vmag_GAIA  Vmag_diff
0  14.170092  14.210405   0.040313  13.707405  13.679211  -0.028194
1  14.429459  14.469780   0.040321  13.909408  13.881211  -0.028197
2  15.769276  15.809591   0.040315  14.431633  14.403441  -0.028192
3  15.908183  15.948498   0.040315  14.502578  14.474386  -0.028192
4  16.115710  16.156027   0.040317  14.577137  14.548944  -0.028193
Original B: 14.170092, Calculated B: 14.210405409479218, Diff: 0.04031340947921791
Original g: 13.854267, Calculated g: 44.31691046244616, Diff: 30.46264346244616
-----
Original B: 14.429459, Calculated B: 14.469779993950855, Diff: 0.04032099395085531
Original g: 14.094252, Calculated g: 44.55689086979871, Diff: 30.46263886979871
-----
Original B: 15.769276, Calculated B: 15.809591146401054, Diff: 0.04031514640105449
Original g: 15.095469, Calculated g: 45.55810940896048, Diff: 30.462640408960482
-----
Original B: 15

/var/folders/34/3847lqd14j78mfm70_z1c_r00000gn/T/ipykernel_50225/739859529.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gaia_bprp_BV['Bmag_diff'] = gaia_bprp_BV['Bmag_GAIA'] - pd.to_numeric(gaia_bprp_BV['Bmag'], errors='coerce')
/var/folders/34/3847lqd14j78mfm70_z1c_r00000gn/T/ipykernel_50225/739859529.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gaia_bprp_BV['Vmag_diff'] = gaia_bprp_BV['Vmag_GAIA'] - pd.to_numeric(gaia_bprp_BV['Vmag'], errors='coerce')
